In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.base import clone

In [2]:
REPO_PATH = Path('/content/course_paper')

!git clone --depth 1 --filter=blob:none --sparse https://github.com/incRED1bl/course_paper.git {REPO_PATH}
!git -C {REPO_PATH} sparse-checkout set colab/data/respiratory_features.csv

sys.path.insert(0, str(REPO_PATH))

!cp {REPO_PATH}/colab/data/respiratory_features.csv ../../data/respiratory_features.csv

Cloning into '/content/course_paper'...
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 8 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), done.
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 7 (delta 0), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (7/7), 32.36 KiB | 974.00 KiB/s, done.
remote: Enumerating objects: 1, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 1 (delta 0), reused 1 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (1/1), 44.47 KiB | 650.00 KiB/s, done.
cp: cannot create regular file '../../data/respiratory_features.csv': No such file or directory


In [3]:
df = pd.read_csv('/content/course_paper/colab/data/respiratory_features.csv')
df.head()

,filename,patient_id,low_freq_energy,mid_freq_energy,high_freq_energy,whistle_strength,spectral_centroid,peak_frequency,entropy,complexity,diagnosis
0,101_1b1_Al_sc_Meditron.wav,101,0.638772,0.061506,0.299722,0.027502,2317.564152,21.533203,0.815790,0.146400,Unknown
1,101_1b1_Pr_sc_Meditron.wav,101,0.853642,0.026700,0.119657,0.029523,934.166042,0.000000,0.884253,0.097903,Unknown
2,102_1b1_Ar_sc_Meditron.wav,102,0.697796,0.135208,0.166997,0.034458,1253.170052,0.000000,0.898144,0.087052,Healthy
3,103_2b2_Ar_mc_LittC2SE.wav,103,0.658031,0.058343,0.283626,0.027033,2190.655897,0.000000,0.868986,0.110194,Asthma
4,104_1b1_Al_sc_Litt3200.wav,104,0.954556,0.037111,0.008333,0.046673,81.716123,0.000000,0.647283,0.233463,COPD


In [4]:
df = df.drop(columns=['filename', 'patient_id'])
df = df[df['diagnosis'] != 'Unknown']
df.head()

,low_freq_energy,mid_freq_energy,high_freq_energy,whistle_strength,spectral_centroid,peak_frequency,entropy,complexity,diagnosis
2,0.697796,0.135208,0.166997,0.034458,1253.170052,0.0000,0.898144,0.087052,Healthy
3,0.658031,0.058343,0.283626,0.027033,2190.655897,0.0000,0.868986,0.110194,Asthma
4,0.954556,0.037111,0.008333,0.046673,81.716123,0.0000,0.647283,0.233463,COPD
5,0.799876,0.187994,0.012130,0.044284,198.766348,7.8125,0.701699,0.210539,COPD
6,0.885614,0.090175,0.024211,0.015177,211.954439,15.6250,0.637505,0.237087,COPD


In [5]:
def map_diagnosis(x):
    if x == 'Healthy':
        return 'Healthy'
    elif x == 'COPD':
        return 'COPD'
    else:
        return 'Disease'

df['diagnosis'] = df['diagnosis'].apply(map_diagnosis)
df.head()

,low_freq_energy,mid_freq_energy,high_freq_energy,whistle_strength,spectral_centroid,peak_frequency,entropy,complexity,diagnosis
2,0.697796,0.135208,0.166997,0.034458,1253.170052,0.0000,0.898144,0.087052,Healthy
3,0.658031,0.058343,0.283626,0.027033,2190.655897,0.0000,0.868986,0.110194,Disease
4,0.954556,0.037111,0.008333,0.046673,81.716123,0.0000,0.647283,0.233463,COPD
5,0.799876,0.187994,0.012130,0.044284,198.766348,7.8125,0.701699,0.210539,COPD
6,0.885614,0.090175,0.024211,0.015177,211.954439,15.6250,0.637505,0.237087,COPD


In [6]:
X = df.drop(columns=['diagnosis'])
y_raw = df['diagnosis']

In [7]:
le = LabelEncoder()
y = le.fit_transform(y_raw)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [9]:
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = np.asarray(y_train)
y_test  = np.asarray(y_test)

In [27]:
!pip install optuna

import optuna

scorer = make_scorer(f1_score, average='macro')

def objective(trial):
    C = trial.suggest_float("C", 1e-3, 100, log=True)
    tol = trial.suggest_float("tol", 1e-6, 1e-2, log=True)
    max_iter = trial.suggest_int("max_iter", 200, 2000, step=100)

    solver = trial.suggest_categorical("solver", ["lbfgs", "liblinear", "saga"])

    penalty = "l2"
    if solver in ["liblinear", "saga"]:
        penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet"])

    l1_ratio = None
    if penalty == "elasticnet":
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

    class_weight = trial.suggest_categorical("class_weight", [None, "balanced"])

    poly_degree = trial.suggest_int("poly_degree", 1, 2)

    prep = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("poly", PolynomialFeatures(degree=poly_degree, include_bias=False)),
        ("scaler", StandardScaler()),
    ])

    clf = LogisticRegression(
        C=C,
        tol=tol,
        max_iter=max_iter,
        solver=solver,
        penalty=penalty,
        l1_ratio=l1_ratio,
        class_weight=class_weight,
        random_state=42,
        n_jobs=-1
    )

    model = Pipeline([("prep", prep), ("clf", clf)])

    try:
        scores = cross_val_score(
            model,
            X_train,
            y_train,
            cv=StratifiedKFold(n_splits=4, shuffle=True, random_state=42),
            scoring=scorer,
            n_jobs=-1
        )
        return scores.mean()
    except Exception as e:
        print(f"Trial failed: {e}")
        return 0.0


study = optuna.create_study(
    direction="maximize",
    study_name="respiratory_logreg_macrof1",
    storage="sqlite:///respiratory_optuna.db",
    load_if_exists=True
)

study.optimize(
    objective,
    n_trials=60,
    timeout=5400,
    show_progress_bar=True
)

best_result = {
    "best_macro_f1": round(study.best_value, 5),
    **study.best_params
}

print("\nBest macro-F1:", best_result["best_macro_f1"])
print("best_params = {")
for k, v in best_result.items():
    if k == "best_macro_f1":
        print(f"    'best_macro_f1': {v},")
    else:
        if isinstance(v, str):
            print(f"    '{k}': '{v}',")
        elif isinstance(v, (int, float)):
            print(f"    '{k}': {v},")
        else:
            print(f"    '{k}': {v!r},")
print("}")


[I 2026-02-21 13:27:33,057] Using an existing study with name 'respiratory_logreg_macrof1' instead of creating a new one.


  0%|          | 0/60 [00:00<?, ?it/s]

[I 2026-02-21 13:27:33,431] Trial 120 finished with value: 0.6230821211400954 and parameters: {'C': 0.3770661983152865, 'tol': 0.00010791286922622521, 'max_iter': 1800, 'solver': 'liblinear', 'penalty': 'l2', 'class_weight': 'balanced', 'poly_degree': 2}. Best is trial 96 with value: 0.6413089166158259.
[I 2026-02-21 13:27:33,742] Trial 121 finished with value: 0.6413089166158259 and parameters: {'C': 2.089132680160694, 'tol': 2.0207730450104487e-05, 'max_iter': 1800, 'solver': 'liblinear', 'penalty': 'l2', 'class_weight': 'balanced', 'poly_degree': 2}. Best is trial 96 with value: 0.6413089166158259.
[I 2026-02-21 13:27:34,049] Trial 122 finished with value: 0.6413089166158259 and parameters: {'C': 2.035227716760267, 'tol': 2.6779235216014423e-05, 'max_iter': 1800, 'solver': 'liblinear', 'penalty': 'l2', 'class_weight': 'balanced', 'poly_degree': 2}. Best is trial 96 with value: 0.6413089166158259.
[I 2026-02-21 13:27:34,350] Trial 123 finished with value: 0.6383227540172975 and param

In [31]:
best_results = {
    'best_macro_f1': 0.64934,
    'C': 4.33789361568014,
    'tol': 3.8828886584166564e-05,
    'max_iter': 1700,
    'solver': 'liblinear',
    'penalty': 'l2',
    'class_weight': 'balanced',
    'poly_degree': 2,
}

In [32]:
best_poly_degree = best_result['poly_degree']

prep = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=best_poly_degree, include_bias=False)),
    ("scaler", StandardScaler()),
])

In [38]:
best_clf = LogisticRegression(
    C=best_result['C'],
    tol=best_result['tol'],
    max_iter=best_result['max_iter'],
    solver=best_result['solver'],
    penalty=best_result.get('penalty', 'l2'),
    l1_ratio=best_result.get('l1_ratio', None),
    class_weight=best_result.get('class_weight', None),
    random_state=42,
    n_jobs = 1 if best_result['solver'] == 'liblinear' else -1
)

In [39]:
model = Pipeline([
    ("prep", prep),
    ("clf", best_clf)
])

In [40]:
model["prep"].fit_transform(X_train).shape

(734, 44)

In [41]:
model

Pipeline(steps=[('prep',
                 Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                                 ('poly',
                                  PolynomialFeatures(include_bias=False)),
                                 ('scaler', StandardScaler())])),
                ('clf',
                 LogisticRegression(C=4.33789361568014, class_weight='balanced',
                                    max_iter=1700, n_jobs=1, random_state=42,
                                    solver='liblinear',
                                    tol=3.8828886584166564e-05))])

In [42]:
oof_pred = np.zeros(len(y_train), dtype=np.int64)
fold_acc = []
fold_auc_macro = []
fold_auc_weighted = []

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, val_idx) in enumerate(cv.split(X_train, y_train), 1):
    print(f"Fold {fold}/5 ... ", end="")

    m = clone(model)
    m.fit(X_train.iloc[tr_idx], y_train[tr_idx])

    proba = m.predict_proba(X_train.iloc[val_idx])
    pred = m.classes_[np.argmax(proba, axis=1)]

    oof_pred[val_idx] = pred

    acc = accuracy_score(y_train[val_idx], pred)
    auc_macro   = roc_auc_score(y_train[val_idx], proba, multi_class="ovr", average="macro")
    auc_weighted = roc_auc_score(y_train[val_idx], proba, multi_class="ovr", average="weighted")

    fold_acc.append(acc)
    fold_auc_macro.append(auc_macro)
    fold_auc_weighted.append(auc_weighted)

    print(f"ACC={acc:.4f} | AUC(macro)={auc_macro:.4f} | AUC(w)={auc_weighted:.4f}")

Fold 1/5 ... ACC=0.9048 | AUC(macro)=0.9681 | AUC(w)=0.9667
Fold 2/5 ... ACC=0.8503 | AUC(macro)=0.8999 | AUC(w)=0.9119
Fold 3/5 ... ACC=0.8776 | AUC(macro)=0.8974 | AUC(w)=0.9635
Fold 4/5 ... ACC=0.8571 | AUC(macro)=0.8643 | AUC(w)=0.9551
Fold 5/5 ... ACC=0.9110 | AUC(macro)=0.9592 | AUC(w)=0.9716


In [43]:
print("\n" + "─"*70)
print("Cross-validation results (5 folds)")
print("─"*70)
print(f"Accuracy mean     : {np.mean(fold_acc):.4f} ± {np.std(fold_acc):.4f}")
print(f"AUC macro mean    : {np.mean(fold_auc_macro):.4f} ± {np.std(fold_auc_macro):.4f}")
print(f"AUC weighted mean : {np.mean(fold_auc_weighted):.4f} ± {np.std(fold_auc_weighted):.4f}")
print("─"*70)

print("\nOOF Classification Report (full training set):")
print(classification_report(
    y_train,
    oof_pred,
    target_names=le.classes_,
    digits=4
))


──────────────────────────────────────────────────────────────────────
Cross-validation results (5 folds)
──────────────────────────────────────────────────────────────────────
Accuracy mean     : 0.8802 ± 0.0244
AUC macro mean    : 0.9178 ± 0.0396
AUC weighted mean : 0.9538 ± 0.0216
──────────────────────────────────────────────────────────────────────

OOF Classification Report (full training set):
              precision    recall  f1-score   support

        COPD     0.9881    0.9164    0.9509       634
     Disease     0.4771    0.7222    0.5746        72
     Healthy     0.3514    0.4643    0.4000        28

    accuracy                         0.8801       734
   macro avg     0.6055    0.7010    0.6418       734
weighted avg     0.9137    0.8801    0.8930       734



In [44]:
model.fit(X_train, y_train)
test_pred = model.predict(X_test)
test_acc = accuracy_score(y_test, test_pred)
print("Test ACC:", test_acc)

Test ACC: 0.8532608695652174
